# Compare-Map Misc Visualisations


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

COMPARE_OUTPUT_DIR_NAME = "compare-map-150"


def find_compare_output_dir() -> Path:
    candidates: list[Path] = []

    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent)

    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if vscode_notebook:
        candidates.append(Path(vscode_notebook).expanduser().resolve().parent)

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            cwd / "nas" / COMPARE_OUTPUT_DIR_NAME,
            cwd.parent / COMPARE_OUTPUT_DIR_NAME,
        ]
    )

    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*/metrics.jsonl")):
            return candidate

    raise FileNotFoundError(
        "Could not locate compare output directory. "
        f"Set BASE_OUTPUT_DIR manually to the folder containing metrics.jsonl subdirectories. Tried: {candidates}"
    )


BASE_OUTPUT_DIR = find_compare_output_dir()
print(f"Using compare output directory: {BASE_OUTPUT_DIR}")


def latest_run_dir(base_dir: Path = BASE_OUTPUT_DIR) -> Path:
    candidates = sorted(
        [path for path in base_dir.glob("*/metrics.jsonl")],
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No metrics.jsonl files found under {base_dir}. Run nas/compare-track.py first."
        )
    return candidates[-1].parent

RUN_DIR_OVERRIDE: Path | str | None = "f926b9"  # e.g., "compare_20260410T133741"
# RUN_DIR_OVERRIDE: Path | str | None = None


def resolve_run_dir() -> Path:
    if RUN_DIR_OVERRIDE:
        candidate = Path(RUN_DIR_OVERRIDE)
        if not candidate.is_absolute():
            candidate = BASE_OUTPUT_DIR / candidate
        if candidate.is_file():
            candidate = candidate.parent
        if not candidate.exists():
            raise FileNotFoundError(
                f"Override directory {candidate} was not found. Update RUN_DIR_OVERRIDE."
            )
        metrics_file = candidate / "metrics.jsonl"
        if not metrics_file.exists():
            raise FileNotFoundError(
                f"{metrics_file} was not found; choose a directory containing metrics.jsonl."
            )
        return candidate
    return latest_run_dir()

RUN_DIR = resolve_run_dir()
METRICS_FILE = RUN_DIR / "metrics.jsonl"
print(f"Using metrics from: {RUN_DIR}")

with METRICS_FILE.open("r", encoding="utf-8") as fh:
    METRIC_RECORDS = [json.loads(line) for line in fh if line.strip()]

if not METRIC_RECORDS:
    raise RuntimeError(f"{METRICS_FILE} is empty; rerun compare-track.")

def display_map_name(map_name: str) -> str:
    return Path(map_name).name


rows: list[dict[str, object]] = []
for record in METRIC_RECORDS:
    map_name = record["map"]
    for run in record["runs"]:
        rows.append(
            {
                "map": map_name,
                "map_label": display_map_name(map_name),
                "arch": run["label"],
                "rmse": run.get("rmse"),
            }
        )

df = pd.DataFrame(rows)
pivot = df.pivot_table(index="arch", columns="map_label", values="rmse")
pivot = pivot.sort_index()

def default_highlight_arch(index: pd.Index) -> str | None:
    for candidate in (RUN_DIR.name, "arch8"):
        if candidate in index:
            return candidate
    return None


HIGHLIGHT_ARCH = default_highlight_arch(pivot.index)
print(f"Loaded RMSE table with shape {pivot.shape} (arches x maps).")


## Continuous metric tables


In [ ]:
metric_specs = {
    "crosstrack_rmse_m": {
        "title": "Cross-track RMSE",
        "ylabel": "Cross-track RMSE (m)",
        "table_title": "Cross-track RMSE (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_mean_m": {
        "title": "Mean cross-track error",
        "ylabel": "Mean cross-track error (m)",
        "table_title": "Mean cross-track error (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_std_m": {
        "title": "Cross-track variability",
        "ylabel": "Cross-track std (m)",
        "table_title": "Cross-track std (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_max_m": {
        "title": "Maximum cross-track error",
        "ylabel": "Max cross-track error (m)",
        "table_title": "Max cross-track error (m)",
        "ylim_bottom": 0.0,
    },
    "heading_error_rmse_deg": {
        "title": "Heading-error RMSE",
        "ylabel": "Heading-error RMSE (deg)",
        "table_title": "Heading-error RMSE (deg)",
        "ylim_bottom": 0.0,
    },
    "heading_error_max_deg": {
        "title": "Maximum heading error",
        "ylabel": "Max heading error (deg)",
        "table_title": "Max heading error (deg)",
        "ylim_bottom": 0.0,
    },
    "wall_min_distance_m": {
        "title": "Minimum wall clearance",
        "ylabel": "Min wall distance (m)",
        "table_title": "Min wall distance (m)",
        "ylim_bottom": 0.0,
    },
    "wall_min_distance_risk": {
        "title": "Wall clearance risk",
        "ylabel": "Wall clearance risk (normalized)",
        "table_title": "Wall clearance risk (normalized)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_mean_rad_s": {
        "title": "Mean steering-rate effort",
        "ylabel": "Mean steering rate (rad/s)",
        "table_title": "Mean steering rate (rad/s)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_std_rad_s": {
        "title": "Steering-rate variability",
        "ylabel": "Steering-rate std (rad/s)",
        "table_title": "Steering-rate std (rad/s)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_max_rad_s": {
        "title": "Maximum steering-rate spike",
        "ylabel": "Max steering rate (rad/s)",
        "table_title": "Max steering rate (rad/s)",
        "ylim_bottom": 0.0,
    },
    "speed_mean_m_s": {
        "title": "Mean speed",
        "ylabel": "Mean speed (m/s)",
        "table_title": "Mean speed (m/s)",
        "ylim_bottom": 0.0,
    },
    "speed_std_m_s": {
        "title": "Speed variability",
        "ylabel": "Speed std (m/s)",
        "table_title": "Speed std (m/s)",
        "ylim_bottom": 0.0,
    },
}

metric_rows: list[dict[str, object]] = []
for record in METRIC_RECORDS:
    map_name = record["map"]
    for run in record["runs"]:
        row = {"map": map_name, "map_label": display_map_name(map_name), "arch": run["label"]}
        for metric_key in metric_specs:
            row[metric_key] = run.get(metric_key)
        metric_rows.append(row)

metric_df = pd.DataFrame(metric_rows)
if "wall_min_distance_m" in metric_df.columns:
    wall_span = metric_df["wall_min_distance_m"].max() - metric_df["wall_min_distance_m"].min()
    if wall_span == 0:
        metric_df["wall_min_distance_risk"] = 0.5
    else:
        metric_df["wall_min_distance_risk"] = 1.0 - (
            (metric_df["wall_min_distance_m"] - metric_df["wall_min_distance_m"].min()) / wall_span
        )
metric_pivots = {
    metric_key: (
        metric_df
        .pivot_table(index="arch", columns="map_label", values=metric_key)
        .reindex(index=pivot.index, columns=pivot.columns)
    )
    for metric_key in metric_specs
    if metric_key in metric_df.columns and metric_df[metric_key].notna().any()
}

for metric_key, metric_pivot in metric_pivots.items():
    print(metric_specs[metric_key]["table_title"])
    display(
        metric_pivot.style
        .format("{:.4f}")
        .background_gradient(axis=0, cmap="YlGnBu")
        .set_properties(**{"text-align": "center"})
    )


## Connected metric comparisons

Each plot below matches the RMSE comparison style: maps on the x-axis, one line per architecture, and arch8 highlighted in red.


In [ ]:
def plot_connected_metric(metric_pivot: pd.DataFrame, metric_key: str) -> None:
    spec = metric_specs[metric_key]
    maps = list(metric_pivot.columns)
    x_positions = range(len(maps))
    highlight_arch = HIGHLIGHT_ARCH
    has_highlight = highlight_arch in metric_pivot.index

    fig, ax = plt.subplots(figsize=(14, 8))
    other_arches = [
        arch for arch in metric_pivot.index
        if not (has_highlight and arch == highlight_arch)
    ]
    if other_arches:
        muted_scale = np.linspace(0.3, 0.75, len(other_arches))
        for scale, arch in zip(muted_scale, other_arches):
            ax.plot(
                x_positions,
                metric_pivot.loc[arch].values,
                marker="o",
                linewidth=1.2,
                color=plt.cm.Greys(scale),
                alpha=0.85,
                label=arch,
            )
    if has_highlight:
        ax.plot(
            x_positions,
            metric_pivot.loc[highlight_arch].values,
            marker="o",
            linewidth=2.5,
            color="#d62728",
            label=highlight_arch,
        )

    ax.set_xticks(list(x_positions))
    ax.set_xticklabels(maps, rotation=45, ha="right")
    ax.set_xlim(-0.5, len(maps) - 0.5)
    ax.set_ylabel(spec["ylabel"])
    ax.set_xlabel("Map")
    ax.set_title(f"{spec['title']} comparison ({RUN_DIR.name})")
    ax.set_ylim(bottom=spec.get("ylim_bottom", 0.0))
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.legend(title="Architecture", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

for metric_key, metric_pivot in metric_pivots.items():
    plot_connected_metric(metric_pivot, metric_key)


## Trajectory Progress Profiles

Each plot converts the 2D trajectory into cumulative path distance, then shows how `x` and `y` evolve along that distance. The waypoint centerline is shown as a dashed reference.


In [ ]:
from f110_planning.utils.waypoint_utils import load_waypoints


PROGRESS_FRACTION = 0.1
show = False


def cumulative_path_distance(points: np.ndarray) -> np.ndarray:
    if len(points) == 0:
        return np.array([], dtype=float)
    deltas = np.diff(points[:, :2], axis=0)
    step_dist = np.linalg.norm(deltas, axis=1)
    return np.concatenate([[0.0], np.cumsum(step_dist)])


def first_progress_fraction(points: np.ndarray, fraction: float = PROGRESS_FRACTION) -> tuple[np.ndarray, np.ndarray]:
    s = cumulative_path_distance(points)
    if len(s) == 0:
        return s, points
    cutoff = s[-1] * fraction
    keep = s <= cutoff
    if not keep.any():
        keep[0] = True
    return s[keep], points[keep]


def plot_progress_profile(record: dict, run_dir: Path = RUN_DIR) -> None:
    trace_file = record.get("trace_file")
    if not trace_file:
        print(f"[warn] {record['map']} has no trace file; skipping")
        return

    trace_path = run_dir / trace_file
    if not trace_path.exists():
        print(f"[warn] missing trace file {trace_path}; skipping")
        return

    waypoints = load_waypoints(record["waypoints"])
    waypoint_s, waypoint_points = first_progress_fraction(waypoints)
    map_label = display_map_name(record["map"])
    highlight_arch = HIGHLIGHT_ARCH

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    with np.load(trace_path) as traces:
        other_runs = [
            run for run in record["runs"]
            if not (highlight_arch and run["label"] == highlight_arch)
        ]
        muted_scale = np.linspace(0.3, 0.75, max(len(other_runs), 1))

        for scale, run in zip(muted_scale, other_runs):
            key = f"{run['label']}_positions"
            if key not in traces:
                continue
            points = traces[key]
            s, segment = first_progress_fraction(points)
            color = plt.cm.Greys(scale)
            axes[0].plot(s, segment[:, 0], color=color, linewidth=1.0, alpha=0.8, label=run["label"])
            axes[1].plot(s, segment[:, 1], color=color, linewidth=1.0, alpha=0.8, label=run["label"])

        if highlight_arch:
            key = f"{highlight_arch}_positions"
            if key in traces:
                points = traces[key]
                s, segment = first_progress_fraction(points)
                axes[0].plot(s, segment[:, 0], color="#d62728", linewidth=2.4, label=highlight_arch)
                axes[1].plot(s, segment[:, 1], color="#d62728", linewidth=2.4, label=highlight_arch)

    axes[0].plot(waypoint_s, waypoint_points[:, 0], color="#1f77b4", linestyle="--", linewidth=1.6, label="waypoint")
    axes[1].plot(waypoint_s, waypoint_points[:, 1], color="#1f77b4", linestyle="--", linewidth=1.6, label="waypoint")

    axes[0].set_ylabel("x position (m)")
    axes[1].set_ylabel("y position (m)")
    axes[1].set_xlabel("Cumulative path distance (m)")
    axes[0].set_title(f"First {PROGRESS_FRACTION:.0%} trajectory progress | {map_label} ({RUN_DIR.name})")
    for ax in axes:
        ax.grid(True, axis="both", linestyle="--", alpha=0.35)
        ax.legend(title="Run", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

if show:
    for record in METRIC_RECORDS:
        plot_progress_profile(record)


## Cross-Run Metric Scatter

Each x-axis position is a compare run directory. For each run, the dots show the selected metric aggregated across evaluation maps for arch1-7 and the run-specific candidate model, displayed as arch8.


In [ ]:
EXCLUDE_COMPARE_RUN_IDS = set()
TRAINED_MAP_LABELS = {
    "017f7c": "Yas Marina",
    "08e72b": "Moscow Raceway",
    "193215": "Melbourne",
    "1b391f": "Sakhir",
    "3cd867": "Budapest",
    "3d2630": "Oschersleben",
    "4ec05d": "IMS",
    "60de23": "Sao Paulo",
    "782a1c": "Montreal",
    "a8b20f": "Spielberg",
    "be986b": "Austin",
    "d24a66": "Zandvoort",
    "dc559d": "Catalunya",
    "e6278a": "Hockenheim",
    "f847ab": "Brands Hatch",
    "f926b9": "Sepang",
}

def discover_compare_run_dirs(base_dir: Path = BASE_OUTPUT_DIR) -> list[Path]:
    run_dirs = sorted(
        path.parent
        for path in base_dir.glob("*/metrics.jsonl")
        if path.parent.name not in EXCLUDE_COMPARE_RUN_IDS
    )
    if not run_dirs:
        raise FileNotFoundError(f"No metrics.jsonl run directories found under {base_dir}")
    return run_dirs


COMPARE_RUN_DIRS = discover_compare_run_dirs()
print("Compare runs:", ", ".join(path.name for path in COMPARE_RUN_DIRS))
CROSS_RUN_AGG = "mean"


def resolve_compare_run_dir(path_like: str | Path) -> Path:
    path = Path(path_like)
    if not path.is_absolute():
        path = BASE_OUTPUT_DIR / path
    if path.is_file():
        path = path.parent
    metrics_path = path / "metrics.jsonl"
    if not metrics_path.exists():
        raise FileNotFoundError(f"{metrics_path} does not exist")
    return path


def load_run_metric_rows(run_dir: Path, metric_key: str) -> pd.DataFrame:
    source_metric = "wall_min_distance_m" if metric_key == "wall_min_distance_risk" else metric_key
    metrics_path = run_dir / "metrics.jsonl"
    rows: list[dict[str, object]] = []
    with metrics_path.open("r", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh if line.strip()]

    for record in records:
        for run in record["runs"]:
            raw_label = run["label"]
            arch_label = "arch8" if raw_label == run_dir.name else raw_label
            rows.append(
                {
                    "run_dir": run_dir.name,
                    "map": record["map"],
                    "map_label": display_map_name(record["map"]),
                    "arch": arch_label,
                    "raw_label": raw_label,
                    "value": run.get(source_metric),
                }
            )
    return pd.DataFrame(rows)


def apply_derived_metric_transform(metric_df: pd.DataFrame, metric_key: str) -> pd.DataFrame:
    if metric_key != "wall_min_distance_risk":
        return metric_df
    metric_df = metric_df.copy()
    span = metric_df["value"].max() - metric_df["value"].min()
    if span == 0:
        metric_df["value"] = 0.5
    else:
        metric_df["value"] = 1.0 - ((metric_df["value"] - metric_df["value"].min()) / span)
    return metric_df


def aggregate_cross_run_metric(run_dirs: list[str | Path], metric_key: str) -> pd.DataFrame:
    frames = [load_run_metric_rows(resolve_compare_run_dir(run_dir), metric_key) for run_dir in run_dirs]
    metric_df = pd.concat(frames, ignore_index=True)
    metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")
    metric_df = metric_df.dropna(subset=["value"])
    metric_df = apply_derived_metric_transform(metric_df, metric_key)

    if CROSS_RUN_AGG == "mean":
        return metric_df.groupby(["run_dir", "arch"], as_index=False)["value"].mean()
    if CROSS_RUN_AGG == "median":
        return metric_df.groupby(["run_dir", "arch"], as_index=False)["value"].median()
    raise ValueError(f"Unsupported CROSS_RUN_AGG={CROSS_RUN_AGG!r}")


def plot_cross_run_metric(run_dirs: list[str | Path], metric_key: str) -> None:
    agg = aggregate_cross_run_metric(run_dirs, metric_key)
    resolved_dirs = [resolve_compare_run_dir(run_dir) for run_dir in run_dirs]
    run_labels = [run_dir.name for run_dir in resolved_dirs]
    x_labels = [TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_dirs]
    x_lookup = {label: idx for idx, label in enumerate(run_labels)}
    arch_order = [f"arch{i}" for i in range(1, 9)]

    fig = plt.figure(figsize=(12, 7.5), constrained_layout=True)
    grid = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.12, 0.88])
    legend_ax = fig.add_subplot(grid[0])
    ax = fig.add_subplot(grid[1])
    legend_ax.axis("off")

    handles = []
    labels = []
    for arch in arch_order:
        arch_df = agg[agg["arch"] == arch]
        if arch_df.empty:
            continue
        xs = [x_lookup[row.run_dir] for row in arch_df.itertuples()]
        color = "#d62728" if arch == "arch8" else plt.cm.Greys(0.25 + 0.55 * (int(arch[4:]) - 1) / 6)
        size = 82 if arch == "arch8" else 52
        points = ax.scatter(
            xs,
            arch_df["value"],
            color=color,
            s=size,
            alpha=0.9,
            label=arch,
            zorder=3 if arch == "arch8" else 2,
        )
        handles.append(points)
        labels.append(arch)

    metric_label = metric_specs.get(metric_key, {}).get("ylabel", metric_key)
    ax.set_xticks(range(len(run_labels)))
    ax.set_xticklabels(x_labels, rotation=0)
    ax.set_ylabel(f"{CROSS_RUN_AGG.title()} {metric_label}")
    ax.set_xlabel("Training map")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)

    fig.suptitle(f"{CROSS_RUN_AGG.title()} {metric_label} by run")
    legend_ax.legend(
        handles,
        labels,
        loc="center",
        ncol=len(labels),
        frameon=True,
        framealpha=0.95,
        handletextpad=0.35,
        columnspacing=1.1,
        borderpad=0.45,
    )
    plt.show()


for metric_key in metric_specs:
    plot_cross_run_metric(COMPARE_RUN_DIRS, metric_key)
